# 03. Click relevance: структура разметки и пайплайн очистки кликов

Идея: клик query→SKU — **шумный** weak label. Хотим бинарный классификатор

\[
f(query, sku) \rightarrow \{0,1\}
\]

«пара релевантна / нет», обученный на **ручной** разметке 100+ пар; дальше модель доразмечает остальное и фильтрует шелуху для brand/category clf и retrieval.

> Пока разметки нет — этот ноутбук задаёт **схему, семпл кандидатов, инструкцию разметчику и скелет обучения**.


## 0. Setup


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name in {"complex_eda", "notebooks"}:
    ROOT = ROOT.parents[1] if ROOT.name == "complex_eda" else ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_utils import (
    apply_plot_style, ensure_dirs, load_query_clicks, ARTIFACTS_DIR, FIGURES_DIR,
    MVIDEO_RED, DARK_SLATE,
)

ensure_dirs()
apply_plot_style()

LABEL_DIR = ARTIFACTS_DIR / "click_relevance"
LABEL_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATES_PATH = LABEL_DIR / "candidates_to_label.csv"
LABELS_PATH = LABEL_DIR / "labels.csv"
LABELS_JSONL = LABEL_DIR / "labels.jsonl"
README_LABEL = LABEL_DIR / "LABELING_GUIDE.md"

SAMPLE_N = 200_000
print("ROOT:", ROOT)
print("label dir:", LABEL_DIR)


## 1. Зачем это нужно (связь с EDA)

Из `01_methods_eda`:
- ~73–74% кликов: бренд не написан в запросе → учим brand clf на кликах;
- ~17% запросов кликают ≥2 бренда → часть кликов случайна / exploratory;
- позиция и цена есть, `user_id` нет → классический user-RecSys недоступен, но **pairwise relevance** — да.

Ручная разметка 100–300 пар → supervised denoiser → чище train для brand/category и retrieval.


## 2. Как размечать

### Файл для вашей разметки
Основной: `artifacts/click_relevance/labels.csv`  
(дублировать можно в `labels.jsonl` — удобнее для потоковой доразметки).

### Схема колонок

| column | тип | описание |
|---|---|---|
| `pair_id` | str | стабильный id (`hash query+sku_id`) |
| `query` | str | текст запроса |
| `sku_id` | str/int | id товара |
| `sku_name` | str | название из клика |
| `sku_brand_name` | str | бренд клика |
| `sku_position` | int | позиция в выдаче |
| `sku_price` | float | цена |
| `label` | int | **1** = релевантно, **0** = нет, пусто = ещё не размечено |
| `confidence` | int | 1–3 (опц.) насколько уверены |
| `notes` | str | свободный комментарий |
| `annotator` | str | кто разметил |
| `labeled_at` | str | ISO дата |

### Правила решения `label`

Ставьте **1**, если по запросу разумно ожидать этот SKU (тот же intent: категория/бренд/модель/ключевой ATTR).  
Ставьте **0**, если клик явный шум (другая категория, случайный бренд, аксессуар вместо устройства без сигнала в запросе и т.п.).

Примеры:
| query | sku | label |
|---|---|---:|
| `ноутбук asus 16` | ASUS VivoBook 16/512 | 1 |
| `ноутбук asus 16` | Чехол для MacBook | 0 |
| `наушники logitech g pro` | Logitech G PRO X | 1 |
| `холодильник` | Стиральная машина LG | 0 |

### Процесс
1. Запустите ячейку «кандидаты» → появится `candidates_to_label.csv`.
2. Скопируйте строки в `labels.csv` **или** заполняйте `label` прямо в candidates и сохраните как `labels.csv`.
3. Цель первой волны: **≥ 100** пар, лучше **200–300**, баланс примерно 50/50 если возможно (добавляем hard negatives).
4. Не править `pair_id`.


In [ ]:
guide = '''# Click relevance labeling guide

## Files
- `candidates_to_label.csv` — пул пар для разметки (генерируется ноутбуком)
- `labels.csv` — ваши ответы (тот же schema + заполненный `label`)
- `labels.jsonl` — опционально, по одной JSON-записи на строку

## label
- 1 = query и SKU соответствуют одному поисковому намерению
- 0 = клик шумовой / другой intent

## Tips
- Смотрите бренд, категорию в названии, ключевые ATTR в query
- Если сомневаетесь — `confidence=1` и короткий `notes`
- Старайтесь размечать и top-position клики, и случайные negatives
'''
README_LABEL.write_text(guide, encoding="utf-8")
print("wrote", README_LABEL)


## 3. Генерация кандидатов на разметку

Стратификация:
1. **позитивы-кандидаты** — частые запросы, клик в топ-3 позиции;
2. **сомнительные** — тот же query, но другой бренд / позиция ≥ 10;
3. **hard negatives** — случайный SKU к тому же query (для баланса 0).


In [ ]:
clicks = load_query_clicks(n=SAMPLE_N, seed=42, random=True)
df = clicks.copy()
df["query"] = df["query_text"].astype(str).str.strip()
df["brand"] = df["sku_brand_name"].astype(str).str.strip()
df = df[df["query"].str.len() >= 2]

# frequent queries
q_freq = df["query"].str.lower().value_counts()
frequent = set(q_freq[q_freq >= 20].index)

sub = df[df["query"].str.lower().isin(frequent)].copy()

rng = np.random.default_rng(42)

# 1) top-position candidates
top = sub[sub["sku_position"] <= 3].drop_duplicates(["query", "sku_id"]).sample(n=min(80, len(sub)), random_state=42)

# 2) suspicious: deep position
deep = sub[sub["sku_position"] >= 10].drop_duplicates(["query", "sku_id"])
deep = deep.sample(n=min(60, len(deep)), random_state=1) if len(deep) else deep

# 3) hard negatives: random sku for a query
neg_rows = []
queries_sample = sub["query"].drop_duplicates().sample(n=min(60, sub["query"].nunique()), random_state=2)
sku_pool = sub[["sku_id", "sku_name", "brand", "sku_price", "sku_position"]].drop_duplicates("sku_id")
for q in queries_sample:
    true_skus = set(sub.loc[sub["query"] == q, "sku_id"])
    cand = sku_pool[~sku_pool["sku_id"].isin(true_skus)]
    if cand.empty:
        continue
    row = cand.sample(1, random_state=int(rng.integers(0, 1_000_000))).iloc[0]
    neg_rows.append({
        "query": q,
        "sku_id": row["sku_id"],
        "sku_name": row["sku_name"],
        "sku_brand_name": row["brand"],
        "sku_position": row["sku_position"],
        "sku_price": row["sku_price"],
        "candidate_source": "random_negative",
    })

def pack(frame, source):
    out = frame[["query", "sku_id", "sku_name", "brand", "sku_position", "sku_price"]].copy()
    out = out.rename(columns={"brand": "sku_brand_name"})
    out["candidate_source"] = source
    return out

cand = pd.concat([
    pack(top, "top_position"),
    pack(deep, "deep_position"),
    pd.DataFrame(neg_rows),
], ignore_index=True)

cand["pair_id"] = [
    str(abs(hash((str(q).lower(), str(s))))) for q, s in zip(cand["query"], cand["sku_id"])
]
cand["label"] = pd.NA
cand["confidence"] = pd.NA
cand["notes"] = ""
cand["annotator"] = ""
cand["labeled_at"] = ""

# de-dup
cand = cand.drop_duplicates("pair_id").reset_index(drop=True)
cand.to_csv(CANDIDATES_PATH, index=False, encoding="utf-8-sig")
print("candidates:", len(cand), "→", CANDIDATES_PATH)
display(cand["candidate_source"].value_counts().to_frame("n"))
display(cand.head(8))

# seed empty labels file if missing
if not LABELS_PATH.exists():
    seed = cand.head(0).copy()
    seed.to_csv(LABELS_PATH, index=False, encoding="utf-8-sig")
    print("created empty", LABELS_PATH)
else:
    print("labels file already exists:", LABELS_PATH)


## 4. Загрузка вашей разметки (когда появится)

Ячейка безопасно читает `labels.csv` и показывает баланс классов. Пока файл пустой — просто заглушка.


In [ ]:
labels = pd.read_csv(LABELS_PATH) if LABELS_PATH.exists() else pd.DataFrame()
print("rows in labels.csv:", len(labels))
if len(labels) and "label" in labels.columns:
    labeled = labels.dropna(subset=["label"])
    print("labeled:", len(labeled))
    if len(labeled):
        display(labeled["label"].value_counts(dropna=False).to_frame("n"))
        display(labeled.head())
    else:
        print("Файл есть, но label ещё не заполнены — разметьте candidates и сохраните в labels.csv")
else:
    print("Разметки пока нет. Откройте candidates_to_label.csv и заполните labels.csv по гайду.")


## 5. Скелет модели (запустится после ≥50–100 лейблов)

Фичи (черновик):
- char TF-IDF по `query` и `sku_name` + cosine;
- совпадение бренда / токенов;
- `sku_position`, log(price);
- (позже) эмбеддинги.

Таргет: `label ∈ {0,1}`.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack

MIN_LABELS = 50

if len(labels) and labels["label"].notna().sum() >= MIN_LABELS:
    lab = labels.dropna(subset=["label"]).copy()
    lab["label"] = lab["label"].astype(int)
    lab["q"] = lab["query"].astype(str)
    lab["t"] = lab["sku_name"].astype(str)

    # simple overlap features
    def tokset(s):
        return set(str(s).lower().split())
    lab["jaccard"] = [len(tokset(a) & tokset(b)) / max(1, len(tokset(a) | tokset(b))) for a, b in zip(lab["q"], lab["t"])]
    lab["brand_in_query"] = [
        str(b).lower() in str(q).lower() if len(str(b)) >= 2 else False
        for q, b in zip(lab["q"], lab["sku_brand_name"])
    ]

    Xtr, Xte, ytr, yte = train_test_split(lab, lab["label"], test_size=0.25, random_state=42, stratify=lab["label"])
    vec_q = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=20_000)
    vec_t = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=20_000)
    Qtr, Qte = vec_q.fit_transform(Xtr["q"]), vec_q.transform(Xte["q"])
    Ttr, Tte = vec_t.fit_transform(Xtr["t"]), vec_t.transform(Xte["t"])

    # cosine between query/title in a joint space (fit on concat)
    vec_j = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=20_000)
    Jtr = vec_j.fit_transform(Xtr["q"] + " || " + Xtr["t"])
    Jte = vec_j.transform(Xte["q"] + " || " + Xte["t"])

    num_tr = np.c_[Xtr["jaccard"].values, Xtr["brand_in_query"].astype(float).values]
    num_te = np.c_[Xte["jaccard"].values, Xte["brand_in_query"].astype(float).values]

    Xtr_m = hstack([Qtr, Ttr, Jtr, num_tr])
    Xte_m = hstack([Qte, Tte, Jte, num_te])

    clf = LogisticRegression(max_iter=300, class_weight="balanced")
    clf.fit(Xtr_m, ytr)
    pred = clf.predict(Xte_m)
    proba = clf.predict_proba(Xte_m)[:, 1]
    print(classification_report(yte, pred, digits=3))
    try:
        print("ROC-AUC:", round(roc_auc_score(yte, proba), 3))
    except Exception as e:
        print("AUC n/a:", e)
else:
    print(f"Нужно ≥{MIN_LABELS} размеченных строк в {LABELS_PATH}")
    print("Сейчас можно размечать candidates_to_label.csv → labels.csv")


## 6. Как использовать модель дальше (план)

```text
все клики
  → score = P(relevant|query,sku)
  → порог τ (например 0.6)
  → clean_clicks
       → brand/category classifier
       → retrieval positives
       → eval без шелухи
```

Active learning: периодически брать пары с `score ≈ 0.5`, доразмечать руками, дообучать.

### Чеклист первой волны разметки
- [ ] 100+ пар в `labels.csv`
- [ ] есть и 0, и 1
- [ ] есть top_position и random_negative
- [ ] перезапустить секцию 5
- [ ] сохранить `models/click_relevance_logreg.joblib` (добавим, когда появятся лейблы)


In [ ]:
# Быстрая сводка путей
pd.DataFrame([
    {"path": str(CANDIDATES_PATH.relative_to(ROOT)), "role": "пул на разметку"},
    {"path": str(LABELS_PATH.relative_to(ROOT)), "role": "ваши 0/1 лейблы"},
    {"path": str(LABELS_JSONL.relative_to(ROOT)), "role": "опциональный jsonl"},
    {"path": str(README_LABEL.relative_to(ROOT)), "role": "краткий гайд"},
])
